# Fine-tune the dense encoder on NFCorpus (Colab, GPU)

This notebook fine-tunes `BAAI/bge-small-en-v1.5` on the NFCorpus **train** relevance judgements with
`MultipleNegativesRankingLoss` (in-batch negatives, no duplicate queries per batch), then evaluates the
base and fine-tuned encoders on the **test** split with exactly the same code as the CPU results in the repo
(`evidence_ir.pipeline` + `evidence_ir.evaluate`), including the hybrid and hybrid + re-rank systems.

Runtime: *Runtime -> Change runtime type -> T4 GPU*. Hyper-parameters are fixed in advance (one epoch,
batch 64, lr 2e-5, 10% warm-up); the test split is used once, for the final numbers only.

Outputs: `finetune_metrics.json` (download it and commit it to `docs/results/`).

In [ ]:
# Colab ships torchao 0.10, which breaks recent peft/transformers imports; it is not needed here.
!pip uninstall -y -q torchao
# Force-reinstall so a reused runtime never keeps an older copy of the package.
!pip install -q --force-reinstall --no-deps git+https://github.com/rkesseku/clinical-evidence-retrieval.git
!pip install -q "sentence-transformers>=3.0" "datasets>=2.20" fastapi httpx

In [ ]:
# Plain data files from the official BEIR mirror (no Hub loading scripts).
!wget -q -nc https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip
!unzip -q -o nfcorpus.zip
!ls nfcorpus nfcorpus/qrels

In [ ]:
import json, random, time
import numpy as np, torch
from evidence_ir.data import load_beir, load_corpus, load_queries, load_qrels
from evidence_ir.dense import SentenceTransformerEncoder
from evidence_ir.rerank import CrossEncoderScorer
from evidence_ir.pipeline import SearchPipeline
from evidence_ir.evaluate import evaluate
from evidence_ir.metrics import paired_bootstrap

SEED = 13
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

corpus = load_corpus("nfcorpus/corpus.jsonl")
queries = load_queries("nfcorpus/queries.jsonl")
train_qrels = load_qrels("nfcorpus/qrels/train.tsv")
test = load_beir("nfcorpus", "test")
# no leakage: train and test query sets must be disjoint
assert not set(train_qrels) & set(test.queries)
PREFIX = SentenceTransformerEncoder.BGE_QUERY_PREFIX
pairs = [(PREFIX + queries[q], corpus[d]) for q, rels in train_qrels.items() if q in queries
         for d, s in rels.items() if s > 0 and d in corpus]
print(f"{len(pairs):,} (query, relevant doc) training pairs from {len(train_qrels):,} queries")

In [ ]:
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
from sentence_transformers.training_args import BatchSamplers

BASE = "BAAI/bge-small-en-v1.5"
OUT = "bge-small-nfcorpus-ft"   # keep "bge" in the name so the query prefix is applied at eval
model = SentenceTransformer(BASE, device="cuda")
model.max_seq_length = 256
train_ds = Dataset.from_dict({"anchor": [a for a, _ in pairs], "positive": [p for _, p in pairs]})
args = SentenceTransformerTrainingArguments(
    output_dir="ckpt", num_train_epochs=1, per_device_train_batch_size=64, learning_rate=2e-5,
    warmup_ratio=0.1, fp16=True, batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=100, save_strategy="no", seed=SEED, report_to="none")
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds,
                                     loss=losses.MultipleNegativesRankingLoss(model))
t0 = time.time(); trainer.train(); train_min = (time.time() - t0) / 60
model.save_pretrained(OUT)
print(f"training took {train_min:.1f} min")

In [ ]:
scorer = CrossEncoderScorer("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")
results, scores = {}, {}
for label, name in [("base", BASE), ("finetuned", OUT)]:
    enc = SentenceTransformerEncoder(name, device="cuda")
    pipe = SearchPipeline(test.corpus, encoder=enc, scorer=scorer)
    m, s = evaluate(test, pipe, n_boot=2000)
    results[label], scores[label] = m["systems"], s
    print(label, json.dumps(m["systems"], indent=1))

comparisons = []
for system in ["dense", "hybrid", "hybrid+rerank"]:
    for metric in ["ndcg@10", "recall@100", "mrr@10"]:
        r = paired_bootstrap(scores["base"][system][metric], scores["finetuned"][system][metric])
        comparisons.append({"system": system, "metric": metric, **r})
        print(f"{system:14s} {metric:10s} ft-base {r['diff']:+.4f} "
              f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}] p={r['p_one_sided']:.4f}")

In [ ]:
out = {"n_test_queries": len(test.queries), "n_train_pairs": len(pairs),
       "train_minutes": round(train_min, 2),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "hyperparameters": {"epochs": 1, "batch_size": 64, "lr": 2e-5, "warmup_ratio": 0.1,
                           "max_seq_length": 256, "seed": SEED, "loss": "MNRL, no-duplicate batches"},
       "base": results["base"], "finetuned": results["finetuned"],
       "finetuned_minus_base": comparisons}
json.dump(out, open("finetune_metrics.json", "w"), indent=2)
from google.colab import files
files.download("finetune_metrics.json")